Sheet 1.2: PyTorch essentials
=============================

**Authors: Michael Franke, Amir Mohammadpour**


The preceding session was dedicated to understanding, and then implementing, the core of automatic differentiation: how a composed differentiable computation can propagate gradients backwards through itself by repeated application of the chain rule. Working through micrograd made that mechanism explicit — every `Value` node records its local contribution to the derivative, and a single traversal of the computation graph accumulates the full gradient.

PyTorch is, at its foundation, an industrial realisation of the same idea, extended to $n$-dimensional arrays and accelerated hardware. Its primary data structure is the **tensor**, a generalisation of scalars, vectors, and matrices to arbitrary dimension; its autograd engine tracks gradients through operations on those tensors in a manner directly analogous to what `micrograd` did for scalar `Value` objects. This notebook introduces both, along with `nn.Module`, the abstraction PyTorch provides for organising learnable computations — the interface through which every model in this course will be built.

If you want to install PyTorch locally, follow [these instructions](https://pytorch.org/get-started/locally/).

In [1]:
import torch

## 1. Tensors



Tensors are the default data structure used for the representation of
numbers in PyTorch. In mathematics (algebra), a tensor is a
generalization of the concept of a matrix. For our purposes, let&rsquo;s think
of a tensor as basically an $n$-dimensional array of numbers.

For example, a single scalar (a single number) is a zero-dimensional
array. An $n$-dimensional vector is a one-dimensional array of $n$
numbers. An $n \times m$ matrix is a two-dimensional array with $n$
rows and $m$ columns. All of these -scalars, vectors and matrices- are
tensors. But *tensors also include even more high-dimensional objects*.
For instance, an $k \times n \times m$ tensor is a three-dimensional
array, which includes $k$ matrices, each of which has $n$ rows and
$m$ columns. And so on.

Full documentation for the `torch.Tensor` class can be found here:
[https://pytorch.org/docs/stable/tensors.html](https://pytorch.org/docs/stable/tensors.html)

![img](./pics/03-scalars-vectors-matrices-tensors.png){width=300px}

> <strong><span style="color:#D83D2B;">Exercise 1.2.1: Dimensions of tensors</span></strong>
>
> How many dimensions do the following tensors have?
>
> 1. $1$
> 2. $[1,2,3]$
> 3. $[[1,2], [3,4]]$
> 4. $[[1,2], [3,4], [5,6]]$
> 5. $[[[1,2], [3,4], [5,6]]]$

<details>
<summary>Show solution</summary>

> **Exercise 1.2.1: Dimensions of tensors**
>
> 1. 0 Dimensions
> 2. 1 Dimension
> 3. 2 Dimensions
> 4. 2 Dimensions
> 5. 3 Dimensions

</details>

In [2]:
########## Exercise 1.2.1 #################
tensor1 = torch.tensor(1) 
tensor2 = torch.tensor([1,2,3])
tensor3 = torch.tensor([[1,2], [3,4]])
tensor4 = torch.tensor([[1,2], [3,4], [5,6]])
tensor5 = torch.tensor([[[1,2], [3,4], [5,6]]])

print(tensor1)
print(tensor2)
print(tensor3)
print(tensor4)
print(tensor5)

# you can use .shape in order to check the dimensions
print(tensor1.shape)
print(tensor3.shape)

tensor(1)
tensor([1, 2, 3])
tensor([[1, 2],
        [3, 4]])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[[1, 2],
         [3, 4],
         [5, 6]]])
torch.Size([])
torch.Size([2, 2])


### Creating a tensor



The central constructor, `torch.tensor()`, accepts any Python scalar, list, or nested list and infers the shape and dtype from the input; it also accepts NumPy arrays, copying their data into a new tensor. For structured initialization, PyTorch provides factory functions: `torch.zeros()` and `torch.ones()` fill a tensor of the given shape with constants, `torch.full()` fills with an arbitrary value, and `torch.rand()` fills with values drawn uniformly from the unit interval.

In [3]:
a_list = [1, 2, 3, 4]
tensor_from_list = torch.tensor(a_list)
tensor_from_list

tensor([1, 2, 3, 4])

In [4]:
new_tensor = torch.tensor([1, 2, 3, 4])
new_tensor

tensor([1, 2, 3, 4])

In [5]:
tensor_0d = torch.tensor(1)
tensor_0d

tensor(1)

In [6]:
tensor_2d = torch.tensor([[1, 2, 3], [4, 5, 6]])
tensor_2d

tensor([[1, 2, 3],
        [4, 5, 6]])

In [7]:
import numpy as np

np_array = np.zeros((2, 2))
np_array_to_tensor = torch.tensor(np_array)
np_array_to_tensor

tensor([[0., 0.],
        [0., 0.]], dtype=torch.float64)

In [8]:
zeros = torch.zeros((2, 2))
zeros

tensor([[0., 0.],
        [0., 0.]])

In [9]:
ones = torch.ones((2, 3))
ones

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [10]:
filled = torch.full((4, 3), 5)
filled

tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]])

In [11]:
torch.rand((2, 3))

tensor([[0.4639, 0.4427, 0.3681],
        [0.0116, 0.4185, 0.5489]])

Note that we can control which random numbers are generated by setting a random seed as follows.

In [3]:
seed = 123
# this will always yield the same random number generation:
for i in range(5):
    torch.manual_seed(123)
    print(torch.rand((2,3)))

tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665]])
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665]])
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665]])
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665]])
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665]])


Remember that the seed only initializes the random number generator (RNG) at a particular state. After that, every call to a random function will advance the RNG state. In the following cell, we only set the seed once, and then calling `rand()` 5 times will yield 5 different results.

In [4]:
seed = 123
torch.manual_seed(123)
# this will never yield the same random number generation:
for i in range(5):
    print(torch.rand((2,3)))

tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665]])
tensor([[0.1366, 0.1025, 0.1841],
        [0.7264, 0.3153, 0.6871]])
tensor([[0.0756, 0.1966, 0.3164],
        [0.4017, 0.1186, 0.8274]])
tensor([[0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826]])
tensor([[0.2745, 0.6584, 0.2775],
        [0.8573, 0.8993, 0.0390]])


> <strong><span style="color:#D83D2B;">Exercise 1.2.2: Creating tensors</span></strong>
>
> 1. Create a PyTorch tensor storing the following matrices:
>
>       $a = [[1,2], [3,4], [5,6]]$
>
>       $b = [[[1,2], [3,4], [5,6]], [[10,20], [30,40], [50,60]]]$
>
> 2. Create a PyTorch tensor of size $3 \times 2 \times 4$ filled with the number 3.
>
>
>
> 3. Create a PyTorch vector with 6 random numbers (lying between 0 and 1).



<details>
<summary>Show solution</summary>

```
    ########## Exercise 1.2.2 Task 1 #################
    exercise1a = torch.tensor([[1,2],[3,4],[5,6]])
    print(exercise1a)
    exercise1b = torch.tensor([[[1,2],[3,4],[5,6]],[[10,20],[30,40],[50,60]]])
    print(exercise1b)

    ########## Exercise 1.2.2 Task 2 #################
    exercise2 = torch.full((3, 2, 4), 3)
    print(exercise2)

    ########## Exercise 1.2.2 Task 3 #################
    exercise3 = torch.rand((6))
    print(exercise3)
```

</details>

## Row & column vectors



A one-dimensional tensor carries no orientation: it is neither a row vector nor a column vector. When passed to `torch.matmul()`, its role is inferred from position — treated as a column when left-multiplied by a matrix, and as a row when right-multiplying one. Whenever an explicit layout is required — for instance, when a bias must broadcast along a particular axis — the correct representation is a two-dimensional tensor of shape $(n, 1)$ for a column or $(1, n)$ for a row.

In [12]:
vector = torch.tensor([1, 10, 100])
print(vector)
print(vector.size())

tensor([  1,  10, 100])
torch.Size([3])


In [13]:
col_vector = torch.tensor([[1], [10], [100]])
print(col_vector)
print(col_vector.size())

tensor([[  1],
        [ 10],
        [100]])
torch.Size([3, 1])


### Tensor data types



Every tensor has a uniform element type, accessible via `.dtype`, inferred from the data passed to the constructor: floating-point literals produce `torch.float32`, integer literals produce `torch.int64`, and booleans produce `torch.bool`. When a tensor is constructed from values of mixed types, PyTorch promotes all elements to the most expressive type present — following the hierarchy `float32 > int64 > bool` — so `torch.tensor([True, 1])` yields an `int64` tensor and `torch.tensor([1, 1.5])` a `float32` tensor. The choice of dtype has direct memory consequences: `float32` uses 4 bytes per element, while `float16` and `bfloat16` use 2. Pre-trained model weights are frequently stored in reduced precision to lower GPU memory requirements, so explicit dtype inspection and conversion are routine when loading models.

In [14]:
torch.tensor([1.5, 2.1]).dtype

torch.float32

In [15]:
print(torch.tensor(5).dtype)
print(torch.tensor(5.0).dtype)

torch.int64
torch.float32


The `dtype` argument overrides inference:

In [16]:
print(torch.tensor(5, dtype=torch.float64).dtype)
print(torch.tensor(1.0, dtype=torch.bool).dtype)

torch.float64
torch.bool


In [17]:
true = torch.tensor([True, True])
print(true)
print(true.dtype)

tensor([True, True])
torch.bool


In [18]:
mix = torch.tensor([True, 1])
print(mix)
print(mix.dtype)

tensor([1, 1])
torch.int64


PyTorch tensors have no string type. In language modelling, text is always represented as an integer tensor of token IDs: a tokenizer partitions a string into subword units drawn from a fixed vocabulary and maps each unit to its integer index. As a preview of that representation, here is the same idea applied at the character level using ASCII codes — each character maps to an integer, and the whole string becomes a 1D `int64` tensor:

In [19]:
hello = "Hello World!"
hello_tensor = torch.tensor([ord(char) for char in hello])
hello_tensor

tensor([ 72, 101, 108, 108, 111,  32,  87, 111, 114, 108, 100,  33])

### Attributes of a tensor



Three tensor attributes are encountered constantly. The `.dtype` and `.shape` attributes (the latter equivalent to `.size()`) record the element type and the size along each dimension. The `.device` attribute records where the tensor resides — `cpu` by default, or a CUDA device identifier once the tensor has been moved to a GPU. All tensors involved in a computation must reside on the same device; mismatched devices raise an error rather than silently transferring data.

In [20]:
print(f"Datatype of tensor         : {hello_tensor.dtype}")
print(f"Shape of tensor            : {hello_tensor.shape}")
print(f"Device tensor is stored on : {hello_tensor.device}")

Datatype of tensor         : torch.int64
Shape of tensor            : torch.Size([12])
Device tensor is stored on : cpu


> <strong><span style="color:#D83D2B;">Exercise 1.2.3: Tensor attributes & types </span></strong>
>
> 1. Inspect the tensor type with `.dtype` for tensors created from a list containing two different data types supported by PyTorch (int, float, Boolean).
>
> 2. Use `.shape` or `.size()` to inspect the shape of a (row) vector, a single column matrix, and a $2 \times 3$ matrix.

<details>
<summary>Show solution</summary>

The implicit casting hierarchy is `float32 > int64 > bool`: when a tensor is constructed from values of mixed types, PyTorch promotes all elements to the highest-precision type present. The executable solution above demonstrates this for each combination.

```

mix = torch.tensor([True, 1])
print(mix)
print(mix.dtype)
print(mix.shape)

mix2 = torch.tensor([True, 1.2])
print(mix2)
print(mix2.dtype)

mix3 = torch.tensor([1, 1.2])
print(mix3)
print(mix3.dtype)

mix4 = torch.tensor([1.2, 1])
print(mix4)
print(mix4.dtype)

mix5 = torch.tensor([1, True])
print(mix5)
print(mix5.dtype)

mix6 = torch.tensor([1.2, True])
print(mix6)
print(mix6.dtype)

mix7 = torch.tensor([[1.2, True],[2, 1.2],[False, 2]])
print(mix7)
print(mix7.dtype)
print(mix7.shape)

mix8 = torch.tensor([[1.2],[2],[False]])
print(mix8)
print(mix8.dtype)
print(mix8.shape)
print(mix8.size())
```

</details>

## 2. Operations on tensors



### Indexing and slicing



PyTorch tensors support the same indexing and slicing syntax as NumPy arrays, using comma-separated indices for multiple dimensions. The most common patterns in practice involve selecting along the batch or the sequence dimension.

In [21]:
matrix = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(matrix)
print(matrix[1, 2])  # single element
print(matrix[2, :])  # third row
print(matrix[2])  # third row (alternative)
print(matrix[:, 1])  # second column

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
tensor(6)
tensor([7, 8, 9])
tensor([7, 8, 9])
tensor([2, 5, 8])


### Joining tensors



`torch.cat()` concatenates tensors along an existing dimension, leaving the number of dimensions unchanged. `torch.stack()` introduces a new dimension and stacks tensors along it — the typical operation when assembling a batch from individually processed examples.

In [22]:
head = torch.tensor([1, 2, 3])
tail = torch.tensor([4, 5, 6])
head_and_tail = torch.cat([head, tail])
print(head_and_tail)

tensor([1, 2, 3, 4, 5, 6])


In [23]:
print(torch.stack([head, tail]))

tensor([[1, 2, 3],
        [4, 5, 6]])


### Reshaping



`torch.reshape()` returns a tensor of the specified shape, leaving the original unchanged. Passing $-1$ for one dimension instructs PyTorch to infer its size from the total element count and the remaining specified dimensions — a pattern that appears often enough to be worth recognising on sight.

In [24]:
tensor_1 = torch.tensor([[1, 2], [3, 4]])
tensor_2 = tensor_1.reshape(4, 1)
print(tensor_1)
print(tensor_2)

tensor([[1, 2],
        [3, 4]])
tensor([[1],
        [2],
        [3],
        [4]])


In [25]:
a = torch.tensor([[0, 1], [2, 3]])
b = torch.reshape(a, (-1,))  # to vector
c = torch.reshape(a, (-1, 1))  # to one col matrix (~ col vector)
d = torch.reshape(a, (1, -1))  # to one row matrix
print(a)
print(b)
print(c)
print(d)

tensor([[0, 1],
        [2, 3]])
tensor([0, 1, 2, 3])
tensor([[0],
        [1],
        [2],
        [3]])
tensor([[0, 1, 2, 3]])


There are also the functions `.squeeze()` and `.unsqueeze()` that remove or add the dimension of size 1 at a given location of a tensor:

In [26]:
e = b.unsqueeze(1)
print(e)
f = e.unsqueeze(0)
print(f)
# we can remove dimension of size 1 at a given position
print(f.squeeze(0))
# or remove all dimensions of size 1
print(f.squeeze())

tensor([[0],
        [1],
        [2],
        [3]])
tensor([[[0],
         [1],
         [2],
         [3]]])
tensor([[0],
        [1],
        [2],
        [3]])
tensor([0, 1, 2, 3])


There is also the function `.flatten()` which returns all elements of a tensor.



In [27]:
a = torch.tensor([[[0, 1], [2, 3]], [[4, 5], [6, 7]]])
print(a)
print(torch.flatten(a))

tensor([[[0, 1],
         [2, 3]],

        [[4, 5],
         [6, 7]]])
tensor([0, 1, 2, 3, 4, 5, 6, 7])


### Transposing



`torch.transpose(tensor, dim0, dim1)` exchanges two dimensions of a tensor. For tensors of rank three or higher, the most frequent case is swapping the last two dimensions — for example, transposing the sequence and feature dimensions within each head in an attention computation.

In [28]:
tensor_1 = torch.tensor(
    [[[10, 20, 30], [40, 50, 60], [70, 80, 90]], [[1, 2, 3], [4, 5, 6], [7, 8, 9]]]
)
tensor_1_transpose = torch.transpose(tensor_1, 1, 2)
print(tensor_1)
print(tensor_1_transpose)

tensor([[[10, 20, 30],
         [40, 50, 60],
         [70, 80, 90]],

        [[ 1,  2,  3],
         [ 4,  5,  6],
         [ 7,  8,  9]]])
tensor([[[10, 40, 70],
         [20, 50, 80],
         [30, 60, 90]],

        [[ 1,  4,  7],
         [ 2,  5,  8],
         [ 3,  6,  9]]])


### Tensor arithmetic



The usual infix notation for arithmetic functions works element-wise on tensors:



In [29]:
x = torch.tensor([1, 2, 3])
y = torch.tensor([1, 4, 8])
print(x + y)
print(x - y)
print(x * y)
print(x / y)
print(y**x)

tensor([ 2,  6, 11])
tensor([ 0, -2, -5])
tensor([ 1,  8, 24])
tensor([1.0000, 0.5000, 0.3750])
tensor([  1,  16, 512])


### Broadcasting



When we apply these operations to tensors of different sizes, PyTorch will try to broadcast the input.

For example, if we multiply a vector with a scalar, the scalar is broadcasted (extended) to a vector of the same length.
The result is that each element in the vector is multiplied by that scalar.



In [30]:
x = torch.tensor([1, 2, 3])
print(x * 4)

tensor([ 4,  8, 12])


Similarly, for higher dimensions.
With the usual arithmetic operations, a vector will be recycled, e.g., to apply to each row of a matrix.



In [31]:
vector = torch.tensor([1, 10])
matrix = torch.tensor([[1, 2], [3, 4]])
print("multiplication:\n", matrix * vector)

multiplication:
 tensor([[ 1, 20],
        [ 3, 40]])


In [32]:
print("division:\n", matrix / vector)

division:
 tensor([[1.0000, 0.2000],
        [3.0000, 0.4000]])


In [33]:
print("addition:\n", matrix + vector)
print("subtraction:\n", matrix - vector)

addition:
 tensor([[ 2, 12],
        [ 4, 14]])
subtraction:
 tensor([[ 0, -8],
        [ 2, -6]])


The precise documentation of broadcasting is [here](https://pytorch.org/docs/stable/notes/broadcasting.html#broadcasting-semantics).



### Matrix Multiplication



Matrix multiplication is performed using `torch.matmul(tensor1, tensor2)`, or its shorthand `tensor1 @ tensor2`. If `tensor1` has shape $(n \times m)$ and `tensor2` has shape $(m \times p)$, the result has shape $(n \times p)$.



In [34]:
tensor1 = torch.tensor([[1, 2], [3, 4], [5, 6]])
tensor2 = torch.tensor([[10, 0], [1, 100]])
print(torch.matmul(tensor1, tensor2))
print(tensor1 @ tensor2)

tensor([[ 12, 200],
        [ 34, 400],
        [ 56, 600]])
tensor([[ 12, 200],
        [ 34, 400],
        [ 56, 600]])


Notice that the function `torch.matmul()` implicitly converts and broadcasts and so also flexibly yields a dot-product, a matrix-vector product or a vector-matrix product.



In [35]:
matrix = torch.tensor([[1, 2], [3, 4]])
vector = torch.tensor([1, 10])
print(matrix)
print(vector)
print(vector @ vector)  # dot prodcut
print(matrix @ vector)  # matrix-vector product (vector is treated as a column vector)
print(vector @ matrix)  # vector-matrix product (vector is treated as a row vector)


tensor([[1, 2],
        [3, 4]])
tensor([ 1, 10])
tensor(101)
tensor([21, 43])
tensor([31, 42])


Full documentation of `torch.matmul()` is [here](https://pytorch.org/docs/stable/generated/torch.matmul.html).



### Assessing just the values of a tensor



The `tensor.item()` function returns the value of a single-item tensor without any further information, which is often useful for inspection or plotting of results:



In [36]:
tensor = torch.tensor([[1, 2], [3, 4], [5, 6]])
print(tensor[1, 1])
print(tensor[1, 1].item())

tensor(4)
4


To convert a larger tensor back to numpy (e.g., for plotting) you can do this:



In [37]:
another_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])
another_tensor.detach().numpy()

array([[1, 2, 3],
       [4, 5, 6]])

> <strong><span style="color:#D83D2B;">Exercise 1.2.4: Operations on tensors</span></strong>
>
> 1. Define a tensor for matrix $[[[1,2], [3,4], [5,6]]]$. Create new tensors obtained by reshaping this matrix into (1) a vector (row vector), (2) a one-column matrix. Also, create its transpose.
>
> 2. Compute the dot product between $[1,3,5]$ and $[1,10,100]$.
>
> 3. Compute the matrix product between PyTorch tensors $[[1], [2], [3]]$ and $[[1,10,100]]$. Convert the result to a numpy array.



<details>
<summary>Show solution</summary>

```########## Exercise 1.2.4 Task 1 #################

ex1 = torch.tensor([[[1,2],[3,4],[5,6]]])
print(ex1)
# 1
ex1_row = torch.flatten(ex1)
print(ex1_row)
# 2
ex1_col = torch.reshape(ex1, (-1, 1))
print(ex1_col)
ex1_col_trans = torch.transpose(ex1_col, 0, -1)
print(ex1_col_trans)

########## Exercise 1.2.4 Task 2 #################
torch1 = torch.tensor([1,3,5])
torch2 = torch.tensor([1,10,100])
dot = torch1 @ torch2
print(dot)

########## Exercise 1.2.4 Task 3 #################

matrix1 = torch.tensor([[1],[2],[3]])
matrix2 = torch.tensor([[1,10,100]])
matrixProd = matrix1 @matrix2
print(matrixProd.numpy())
```

</details>

## 3. Autograd


The `micrograd` engine you built in the preceding session operated on scalar `Value` objects: each node in the computation graph stored a `data` field, a `grad` field, and a `_backward` closure that propagated the local gradient contribution one step upstream. Calling `.backward()` on the output traversed that graph in reverse topological order, accumulating partial derivatives at every node.

PyTorch's autograd engine applies the same principle to tensors. Any tensor created with `requires_grad=True` participates in gradient tracking: each operation that involves it is recorded in a dynamic computation graph, and calling `.backward()` on a scalar output propagates gradients to all upstream tensors, depositing the accumulated partial derivative in their `.grad` attribute. The correspondence to micrograd is direct:

| micrograd | PyTorch |
|:----------|:--------|
| `Value(x)` | `torch.tensor(x, requires_grad=True)` |
| `v.data` | `t.item()` or `t.detach()` |
| `v.grad` | `t.grad` |
| `v.backward()` | `loss.backward()` |

In [38]:
# A simple computation with gradient tracking.
# z = x^2 + y*x, so dz/dx = 2x + y and dz/dy = x.
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)

z = x**2 + y * x
z.backward()  # propagate gradients

print(f"z        = {z.item():.1f}")        # 9 + 12 = 21
print(f"dz/dx    = {x.grad.item():.1f}")   # 2*3 + 4 = 10
print(f"dz/dy    = {y.grad.item():.1f}")   # 3

z        = 21.0
dz/dx    = 10.0
dz/dy    = 3.0


One important practical detail: gradients **accumulate** in `.grad` across successive calls to `.backward()`. In a training loop, it is therefore necessary to zero the gradients before each backward pass — either through an optimizer (`optimizer.zero_grad()`) or directly on the tensor (`x.grad.zero_()`).

When computing quantities that should not influence the computation graph — for instance, during evaluation or when inspecting outputs — wrap the code in `torch.no_grad()` to suppress gradient tracking entirely:

In [39]:
with torch.no_grad():
    val = x**2 + y * x
    print(val)   # forward pass executed, but no graph is recorded

tensor(21.)


## 4. Modules: `nn.Module`


Tensors with `requires_grad=True` and a manual call to `.backward()` are sufficient, in principle, to implement any neural network: parameters are tensors, the forward pass is Python, and the backward pass fills their `.grad` attributes for a manual update step. The notebooks that follow will ask you to do exactly this — implement components from scratch, in that form, before revealing PyTorch's higher-level equivalent.

That equivalent is `nn.Module`: a base class that imposes a small interface on any learnable computation. The interface has two obligations:

1. **`__init__`** — declare parameters and submodules, registering them so that PyTorch can find them automatically.
2. **`forward`** — define the computation; this method is called when the module is invoked as a function.

Everything else — collecting parameters for an optimizer, moving a model to a device, saving and loading weights — is handled by inherited methods. Here is the pattern, applied to the single most basic case: a linear transformation $y = xW^\top + b$.

In [40]:
import torch.nn as nn

class LinearLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # Wrapping a tensor in nn.Parameter registers it with the module,
        # so .parameters() can find it and requires_grad is set automatically.
        self.W = nn.Parameter(torch.randn(out_features, in_features))
        self.b = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        return x @ self.W.T + self.b

layer = LinearLayer(in_features=4, out_features=2)
x = torch.randn(3, 4)         # batch of 3 inputs, each of dimension 4
out = layer(x)                # invokes forward internally
print(out.shape)              # torch.Size([3, 2])
print([p.shape for p in layer.parameters()])

torch.Size([3, 2])
[torch.Size([2, 4]), torch.Size([2])]


PyTorch's built-in `nn.Linear` is the same thing — identical weight and bias parameters, identical `forward` logic — with additional options for initialization and bias suppression:

In [41]:
layer_builtin = nn.Linear(in_features=4, out_features=2)
out_builtin = layer_builtin(x)
print(out_builtin.shape)

torch.Size([3, 2])


Every component in this course follows this same interface: subclass `nn.Module`, register parameters in `__init__`, define the computation in `forward`. You will implement each one from scratch first, in exactly the form shown above, before the built-in equivalent is revealed.